# Experiment 2: CA-Mixer vs MLP-Mixer vs ResMLP on CIFAR-100 (matched baselines)
Single file for Google Colab / Kaggle (free tier, T4). Run the cells top to bottom.

**Model families (every run records its full config):**
* `MLP-Mixer`, `ResMLP`: *base* (small), *param-matched* (just over CA-Mixer's parameter count),
  *FLOP-matched* (~0.5 GFLOPs). Matching is done automatically by searching the channel-MLP width.
* `CA-Mixer`: spec (kappa=1, T=max(h,w)), efficient (kappa=0.5 + T-jitter), multi-resolution training.
* Optional ablations/controls: no jitter, fixed T, T=1, no mask, learnable filters, identity kernel, DWConv-Mixer.

**Outputs** (in the results folder): `table_configs.csv`, `summary.csv`, `summary_numeric.csv`, `table_main.tex`,
`table_matched.csv`, `table_resolution.csv`, `table_tsweep.csv`, `table_best_T.csv`, `table_complexity.csv`,
figures (`fig_*.png`), `env.json`, a zip of everything. Finished runs are cached -> re-running skips them.

In [ ]:
# Config (edit here)
N_SEEDS = 1                        # <<< pilot with 1 seed. Later set 5: only the missing seeds are trained.
RUN_GROUPS = ["main", "ca_extra"]  # "main" | "ca_extra" | "ablation"
ONLY = None                        # run only these names, e.g. ["CA-Mixer (spec, kappa=1)", "ResMLP (FLOP-matched)"]
QUICK = False                      # True = smoke test (2 epochs, tiny sweeps). Run this first!
RECIPE = "basic"                   # "basic" or "cutmix" (same recipe for every model)
LR_PER_MODEL = {}                  # optional manual tuning, e.g. {"ResMLP (FLOP-matched)": 2e-3}
CIFAR_PATH = None                  # path to your cifar-100-python.tar.gz OR extracted folder (auto-detected if None)
USE_DRIVE = True                   # Colab: mount Drive (read your CIFAR file from it, keep results/checkpoints on it)
RESULTS_FOLDER = "ca_mixer_exp2"   # created inside MyDrive (Colab) or /kaggle/working
TARGET_GFLOPS = 0.5                # FLOP-matched baselines target
RUN_RESOLUTION = True              # accuracy at other test resolutions (native + resize->32 adapter)
RUN_TSWEEP = True                  # CA models: accuracy for a grid of (resolution x T), inference only
RUN_COMPLEXITY = True              # params/FLOPs/latency/memory vs resolution
SAVE_CKPT = True
RES_LIST = [16, 24, 32, 40, 48, 64]
T_LIST = [2, 3, 4, 5, 6, 8, 10, 12, 16, 24]   # includes the nominal T of kappa=0.5 and kappa=1 at every RES_LIST entry
COMPLEXITY_RES = [32, 64, 96, 128]
TSWEEP_N = 5000                    # test images used for the T-sweep
TSWEEP_SEEDS = [0]                 # T-sweep is a diagnostic: only these seeds run it (saves ~2-3 min per CA run per seed)

CFG = dict(epochs=30, bs=256, lr=1e-3, wd=0.05, warmup=2, ls=0.1, clip=1.0, amp=True,
           cutmix_prob=0.0, cutmix_alpha=1.0)
if RECIPE == "cutmix":
    CFG["cutmix_prob"] = 0.5
if QUICK:
    CFG["epochs"] = 2; N_SEEDS = 1
    RES_LIST = [24, 32, 48]; T_LIST = [2, 4, 8]; COMPLEXITY_RES = [32, 64]; TSWEEP_N = 1000
SEEDS = list(range(N_SEEDS))

In [ ]:
# Imports & environment
import os, sys, time, json, math, random, shutil, hashlib, glob, pickle, tarfile
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if dev.type != "cuda":
    print("WARNING: no GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2.")
AMP = bool(CFG["amp"]) and dev.type == "cuda"
torch.backends.cudnn.benchmark = True

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_COLAB:
    OUT = f"/content/{RESULTS_FOLDER}"
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            OUT = f"/content/drive/MyDrive/{RESULTS_FOLDER}"
        except Exception as e:
            print("Drive mount failed, results stay on the temporary VM disk:", e)
elif IN_KAGGLE:
    OUT = f"/kaggle/working/{RESULTS_FOLDER}"
else:
    OUT = f"./{RESULTS_FOLDER}"
RUNS, CKPT = os.path.join(OUT, "runs"), os.path.join(OUT, "ckpt")
for d in (OUT, RUNS, CKPT):
    os.makedirs(d, exist_ok=True)
if IN_KAGGLE:   # resume: reuse finished runs from a previous notebook version attached via Add Data > Notebook output
    for f in glob.glob(f"/kaggle/input/**/{RESULTS_FOLDER}/runs/*.json", recursive=True):
        if not os.path.exists(os.path.join(RUNS, os.path.basename(f))): shutil.copy(f, RUNS)
DATA_DIR = "/kaggle/working/data" if IN_KAGGLE else ("/content/data" if IN_COLAB else "./data")
print("torch", torch.__version__, "| device", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "-")
print("results ->", OUT)

In [ ]:
# Helpers (pure python, no torch)
def arch_str(a):
    f = a["family"]; base = f"patch={a['patch']} dim={a['dim']} depth={a['depth']}"
    if f == "mixer":
        return f"MLP-Mixer | {base} token_hidden={a['tok']} channel_hidden={a['ch']}"
    if f == "resmlp":
        return f"ResMLP | {base} channel_hidden={a['ch']} layerscale_init=0.1"
    if f == "dwconv":
        return f"DWConv-Mixer | {base} channel_hidden={a['ch']} dw_kernel={a['k']}"
    tm = (f"T=ceil({a['kappa']}*max(h,w)), min {a['tmin']}" if a["t_mode"] == "adaptive" else f"T={a['t_mode']} fixed")
    return (f"CA-Mixer | {base} nca_hidden={a['f_hidden']} channel_hidden={a['ch']} | {tm} | T_jitter={a['tjit']} | "
            f"update_prob={a['fire_rate']} | frozen_filters={not a['learn_filters']} | identity_kernel={a['identity']}")

def recipe_str(spec):
    lr = LR_PER_MODEL.get(spec["name"], CFG["lr"])
    return (f"epochs={CFG['epochs']} bs={CFG['bs']} lr={lr:g} wd={CFG['wd']} label_smoothing={CFG['ls']} "
            f"cutmix_p={CFG['cutmix_prob']} train_res={spec['train'].get('train_res', '32 only')} AMP={AMP}")

def to_tex(d, caption):
    cs = "l" + "c" * (d.shape[1] - 1)
    L = [r"\begin{table}[t]\centering\small", rf"\caption{{{caption}}}", rf"\begin{{tabular}}{{{cs}}}", r"\toprule",
         " & ".join(d.columns).replace("%", r"\%") + r" \\", r"\midrule"]
    L += [" & ".join(str(v) for v in r).replace("±", r"$\pm$").replace("%", r"\%") + r" \\" for r in d.values]
    return "\n".join(L + [r"\bottomrule", r"\end{tabular}", r"\end{table}"])

In [ ]:
# Models
class LN2d(nn.Module):
    """Per-position LayerNorm over channels (no cross-position statistics -> resolution agnostic)."""
    def __init__(s, c):
        super().__init__(); s.ln = nn.LayerNorm(c)
    def forward(s, x):
        return s.ln(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

class Affine(nn.Module):
    def __init__(s, d):
        super().__init__(); s.alpha = nn.Parameter(torch.ones(d)); s.beta = nn.Parameter(torch.zeros(d))
    def forward(s, x):
        return x * s.alpha + s.beta

# ---------------- MLP-Mixer ----------------
class MixerBlock(nn.Module):
    def __init__(s, n, d, dt, dc):
        super().__init__()
        s.n1 = nn.LayerNorm(d); s.t1 = nn.Linear(n, dt); s.t2 = nn.Linear(dt, n)
        s.n2 = nn.LayerNorm(d); s.c1 = nn.Linear(d, dc); s.c2 = nn.Linear(dc, d)
    def forward(s, x):                                   # B,N,D
        y = s.n1(x).transpose(1, 2)
        x = x + s.t2(F.gelu(s.t1(y))).transpose(1, 2)
        return x + s.c2(F.gelu(s.c1(s.n2(x))))

class MLPMixer(nn.Module):
    def __init__(s, img=32, patch=4, dim=128, depth=6, tok=256, ch=512, nc=100):
        super().__init__()
        s.embed = nn.Conv2d(3, dim, patch, patch); n = (img // patch) ** 2
        s.blocks = nn.Sequential(*[MixerBlock(n, dim, tok, ch) for _ in range(depth)])
        s.norm = nn.LayerNorm(dim); s.head = nn.Linear(dim, nc)
    def forward(s, x):
        x = s.embed(x).flatten(2).transpose(1, 2)
        return s.head(s.norm(s.blocks(x)).mean(1))

# ---------------- ResMLP (Touvron et al.): Affine + one NxN linear + LayerScale ----------------
class ResMLPBlock(nn.Module):
    def __init__(s, n, d, ch, init=0.1):
        super().__init__()
        s.n1 = Affine(d); s.tok = nn.Linear(n, n); s.g1 = nn.Parameter(init * torch.ones(d))
        s.n2 = Affine(d); s.c1 = nn.Linear(d, ch); s.c2 = nn.Linear(ch, d); s.g2 = nn.Parameter(init * torch.ones(d))
    def forward(s, x):                                   # B,N,D
        x = x + s.g1 * s.tok(s.n1(x).transpose(1, 2)).transpose(1, 2)
        return x + s.g2 * s.c2(F.gelu(s.c1(s.n2(x))))

class ResMLP(nn.Module):
    def __init__(s, img=32, patch=4, dim=128, depth=6, ch=512, nc=100):
        super().__init__()
        s.embed = nn.Conv2d(3, dim, patch, patch); n = (img // patch) ** 2
        s.blocks = nn.Sequential(*[ResMLPBlock(n, dim, ch) for _ in range(depth)])
        s.norm = Affine(dim); s.head = nn.Linear(dim, nc)
    def forward(s, x):
        x = s.embed(x).flatten(2).transpose(1, 2)
        return s.head(s.norm(s.blocks(x)).mean(1))

# ---------------- DWConv-Mixer (local-conv control) ----------------
class DWBlock(nn.Module):
    def __init__(s, d, ch, k):
        super().__init__()
        s.n1 = LN2d(d); s.dw = nn.Conv2d(d, d, k, padding=k // 2, groups=d, padding_mode="replicate")
        s.n2 = LN2d(d); s.c1 = nn.Conv2d(d, ch, 1); s.c2 = nn.Conv2d(ch, d, 1)
    def forward(s, x):
        x = x + s.dw(s.n1(x))
        return x + s.c2(F.gelu(s.c1(s.n2(x))))

class DWConvMixer(nn.Module):
    def __init__(s, patch=4, dim=128, depth=6, ch=512, k=7, nc=100):
        super().__init__()
        s.embed = nn.Conv2d(3, dim, patch, patch)
        s.blocks = nn.Sequential(*[DWBlock(dim, ch, k) for _ in range(depth)])
        s.norm = LN2d(dim); s.head = nn.Linear(dim, nc)
    def forward(s, x):
        return s.head(s.norm(s.blocks(s.embed(x))).mean((2, 3)))

# ---------------- CA-Mixer ----------------
def perception_kernels(identity=False):
    sx = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]])
    sy = torch.tensor([[-1., -2, -1], [0, 0, 0], [1, 2, 1]])
    lp = torch.tensor([[0., 1, 0], [1, -4, 1], [0, 1, 0]])
    ks = [sx, sy, lp]
    if identity:
        ks = [torch.tensor([[0., 0, 0], [0, 1, 0], [0, 0, 0]])] + ks
    return torch.stack(ks)

class CATokenMixer(nn.Module):
    def __init__(s, dim, hidden, fire_rate=0.8, t_mode="adaptive", learn_filters=False, identity=False,
                 kappa=0.5, tmin=2, tjit=0.25):
        super().__init__()
        s.dim, s.fire, s.t_mode, s.kappa, s.tmin, s.tjit = dim, fire_rate, t_mode, kappa, tmin, tjit
        k = perception_kernels(identity)
        s.w = nn.Parameter(k.repeat(dim, 1, 1).unsqueeze(1), requires_grad=learn_filters)   # (K*dim,1,3,3)
        s.f1 = nn.Conv2d(k.shape[0] * dim, hidden, 1)                                        # shared MLP f_theta
        s.f2 = nn.Conv2d(hidden, dim, 1)
        nn.init.zeros_(s.f2.weight); nn.init.zeros_(s.f2.bias)
    def nominal_T(s, h, w):
        if s.t_mode == "adaptive":
            return max(s.tmin, math.ceil(s.kappa * max(h, w) - 1e-9))
        return int(s.t_mode)
    def steps(s, x):
        T = s.nominal_T(*x.shape[-2:])
        if s.training and s.tjit > 0:
            T = max(1, round(T * random.uniform(1 - s.tjit, 1 + s.tjit)))
        return T
    def forward(s, h):
        for _ in range(s.steps(h)):
            p = F.conv2d(F.pad(h, (1, 1, 1, 1), mode="replicate"), s.w, groups=s.dim)
            d = torch.tanh(s.f2(F.gelu(s.f1(p))))
            if s.fire < 1.0:
                if s.training:
                    m = (torch.rand(h.size(0), 1, h.size(2), h.size(3), device=h.device) < s.fire).to(d.dtype)
                    d = d * m
                else:
                    d = d * s.fire
            h = h + d
        return h

class CABlock(nn.Module):
    def __init__(s, dim, f_hidden, ch, **kw):
        super().__init__()
        s.n1 = LN2d(dim); s.mix = CATokenMixer(dim, f_hidden, **kw)
        s.n2 = LN2d(dim); s.c1 = nn.Conv2d(dim, ch, 1); s.c2 = nn.Conv2d(ch, dim, 1)
    def forward(s, x):
        n = s.n1(x)
        x = x + (s.mix(n) - n)
        return x + s.c2(F.gelu(s.c1(s.n2(x))))

class CAMixer(nn.Module):
    def __init__(s, patch=4, dim=128, depth=6, f_hidden=128, ch=512, nc=100, **kw):
        super().__init__()
        s.embed = nn.Conv2d(3, dim, patch, patch)
        s.blocks = nn.ModuleList([CABlock(dim, f_hidden, ch, **kw) for _ in range(depth)])
        s.norm = LN2d(dim); s.head = nn.Linear(dim, nc); s.patch = patch
    def T_at(s, res):
        g = res // s.patch
        return s.blocks[0].mix.nominal_T(g, g)
    def forward(s, x):
        x = s.embed(x)
        for b in s.blocks:
            x = b(x)
        return s.head(s.norm(x).mean((2, 3)))

def build_arch(a, img=32):
    f = a["family"]
    if f == "mixer":  return MLPMixer(img, a["patch"], a["dim"], a["depth"], a["tok"], a["ch"])
    if f == "resmlp": return ResMLP(img, a["patch"], a["dim"], a["depth"], a["ch"])
    if f == "dwconv": return DWConvMixer(a["patch"], a["dim"], a["depth"], a["ch"], a["k"])
    return CAMixer(a["patch"], a["dim"], a["depth"], a["f_hidden"], a["ch"], fire_rate=a["fire_rate"],
                   t_mode=a["t_mode"], learn_filters=a["learn_filters"], identity=a["identity"],
                   kappa=a["kappa"], tmin=a["tmin"], tjit=a["tjit"])

def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def count_flops(model, res=32):
    """Forward GFLOPs/img (matmul + conv only, 2*MACs), via PyTorch FlopCounterMode. Works on CPU or GPU."""
    try:
        from torch.utils.flop_counter import FlopCounterMode
        was = model.training; model.eval(); d = next(model.parameters()).device
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            model(torch.randn(1, 3, res, res, device=d))
        model.train(was)
        return fc.get_total_flops() / 1e9
    except Exception as e:
        print("FLOP count failed:", e); return float("nan")

In [ ]:
# Model registry: automatic parameter / FLOP matching + config table
def bsearch(lo, hi, pred):
    """Smallest x in [lo, hi] with pred(x) True (pred must be monotone False -> True)."""
    while lo < hi:
        mid = (lo + hi) // 2
        if pred(mid): hi = mid
        else: lo = mid + 1
    return lo

def match_params(arch, target):
    """Smallest channel-MLP width whose trainable params are >= target ('just over')."""
    ch = bsearch(16, 8192, lambda c: n_params(build_arch(dict(arch, ch=c))) >= target)
    return dict(arch, ch=ch)

def match_flops(arch, target_g):
    """Channel-MLP width whose GFLOPs/img (at 32 px) is closest to target."""
    f = lambda c: count_flops(build_arch(dict(arch, ch=c)), 32)
    hi = bsearch(16, 8192, lambda c: f(c) >= target_g)
    return dict(arch, ch=min([hi - 1, hi], key=lambda c: abs(f(c) - target_g)))

BASE = dict(patch=4, dim=128, depth=6)
def ca_arch(**kw):
    a = dict(family="ca", **BASE, f_hidden=128, ch=512, fire_rate=0.8, t_mode="adaptive", learn_filters=False,
             identity=False, kappa=0.5, tmin=2, tjit=0.25)
    a.update(kw); return a

MIX_BASE = dict(family="mixer", **BASE, tok=256, ch=512)
RES_BASE = dict(family="resmlp", **BASE, ch=512)
WIDE = dict(patch=4, dim=256, depth=6)                                    # FLOP-matched models are wider (dim 256)
CA_SPEC = ca_arch(kappa=1.0, tmin=1, tjit=0.0)                            # exactly the spec: T = max(h, w)
CA_EFF = ca_arch()                                                         # kappa=0.5 + T-jitter
P_TARGET = n_params(build_arch(CA_SPEC))

SPECS = []
def add(name, role, group, arch, train=None):
    SPECS.append(dict(name=name, role=role, group=group, arch=arch, train=train or {}))

add("MLP-Mixer (base)", "reference", "main", MIX_BASE)
add("ResMLP (base)", "reference", "main", RES_BASE)
add("MLP-Mixer (param-matched)", "param-matched", "main", match_params(MIX_BASE, P_TARGET))
add("ResMLP (param-matched)", "param-matched", "main", match_params(RES_BASE, P_TARGET))
add("MLP-Mixer (FLOP-matched)", "FLOP-matched", "main", match_flops(dict(MIX_BASE, **WIDE), TARGET_GFLOPS))
add("ResMLP (FLOP-matched)", "FLOP-matched", "main", match_flops(dict(RES_BASE, **WIDE), TARGET_GFLOPS))
add("CA-Mixer (spec, kappa=1)", "proposed", "main", CA_SPEC)
add("CA-Mixer (kappa=0.5, T-jitter)", "proposed", "main", CA_EFF)
add("CA-Mixer (kappa=0.5, T-jitter, multi-res train)", "proposed", "ca_extra", CA_EFF, dict(train_res=[24, 32, 40]))
add("CA-Mixer (kappa=0.5, no jitter)", "ablation", "ablation", ca_arch(tjit=0.0))
add("CA-Mixer (T=4 fixed)", "ablation", "ablation", ca_arch(t_mode=4, tjit=0.0))
add("CA-Mixer (T=1)", "ablation", "ablation", ca_arch(t_mode=1, tjit=0.0))
add("CA-Mixer (no mask)", "ablation", "ablation", ca_arch(fire_rate=1.0))
add("CA-Mixer (learnable filters)", "ablation", "ablation", ca_arch(learn_filters=True))
add("CA-Mixer (+identity kernel)", "ablation", "ablation", ca_arch(identity=True))
add("DWConv-Mixer (param-matched, k=7)", "control", "ablation",
    match_params(dict(family="dwconv", **BASE, ch=512, k=7), P_TARGET))

SPEC_BY_NAME = {sp["name"]: sp for sp in SPECS}
if ONLY:
    bad = [n for n in ONLY if n not in SPEC_BY_NAME]
    if bad: print("WARNING: unknown names in ONLY:", bad, "\nValid names:", list(SPEC_BY_NAME))
    ACTIVE = [sp for sp in SPECS if sp["name"] in ONLY]
else:
    ACTIVE = [sp for sp in SPECS if sp["group"] in RUN_GROUPS]

reg = []
for sp in SPECS:
    m = build_arch(sp["arch"])
    reg.append(dict(name=sp["name"], role=sp["role"], group=sp["group"], selected=sp in ACTIVE, config=arch_str(sp["arch"]),
                    recipe=recipe_str(sp), params_M=round(n_params(m) / 1e6, 4), gflops_32px=round(count_flops(m, 32), 4),
                    T_32px=m.T_at(32) if hasattr(m, "T_at") else None))
    del m
reg = pd.DataFrame(reg); reg.to_csv(os.path.join(OUT, "table_configs.csv"), index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 120)
print(reg[["name", "role", "selected", "params_M", "gflops_32px", "T_32px"]].to_string(index=False))
print(f"\nCA-Mixer (spec) params = {P_TARGET/1e6:.4f}M  -> param-matched baselines must be >= this; FLOP target = {TARGET_GFLOPS} GFLOPs")
for r in reg.itertuples():
    if r.role == "param-matched" and r.params_M * 1e6 < P_TARGET: print("WARNING: not param-matched:", r.name)
    if r.role == "FLOP-matched" and abs(r.gflops_32px - TARGET_GFLOPS) > 0.03 * TARGET_GFLOPS: print("WARNING: FLOPs off target:", r.name)
json.dump(dict(date=time.strftime("%Y-%m-%d %H:%M:%S"), torch=torch.__version__,
               gpu=torch.cuda.get_device_name(0) if dev.type == "cuda" else None, CFG=CFG, N_SEEDS=N_SEEDS,
               RUN_GROUPS=RUN_GROUPS, ONLY=ONLY, RECIPE=RECIPE, LR_PER_MODEL=LR_PER_MODEL, TARGET_GFLOPS=TARGET_GFLOPS,
               RES_LIST=RES_LIST, T_LIST=T_LIST), open(os.path.join(OUT, "env.json"), "w"), indent=1)

In [ ]:
# Data: use a local copy if one exists (fast), download only as a last resort (slow)
def _is_cifar_dir(d):
    return os.path.isdir(d) and all(os.path.exists(os.path.join(d, f)) for f in ("train", "test", "meta"))

def _extract(tar_path):
    os.makedirs(DATA_DIR, exist_ok=True)
    print("extracting", tar_path, "->", DATA_DIR)
    with tarfile.open(tar_path) as t:
        t.extractall(DATA_DIR)
    return os.path.join(DATA_DIR, "cifar-100-python")

def locate_cifar_dir():
    if CIFAR_PATH:                                                    # explicit path: never silently download
        p = CIFAR_PATH
        if _is_cifar_dir(p): return p
        if os.path.isdir(p) and _is_cifar_dir(os.path.join(p, "cifar-100-python")): return os.path.join(p, "cifar-100-python")
        if os.path.isfile(p):
            d = _extract(p)
            if _is_cifar_dir(d): return d
        raise FileNotFoundError(f"CIFAR_PATH={p} is not a cifar-100-python.tar.gz or an extracted cifar-100-python folder")
    cands = [os.path.join(DATA_DIR, "cifar-100-python"), "/content/cifar-100-python.tar.gz",
             "/content/drive/MyDrive/cifar-100-python.tar.gz", "/content/drive/MyDrive/cifar-100-python", "./cifar-100-python.tar.gz"]
    for pat in ("/kaggle/input/**/cifar-100-python.tar.gz", "/kaggle/input/**/cifar-100-python.tar",
                "/kaggle/input/**/cifar-100-python", "/kaggle/input/**/meta"):
        cands += glob.glob(pat, recursive=True)
    for c in cands:
        if os.path.isfile(c) and os.path.basename(c) == "meta": c = os.path.dirname(c)
        if _is_cifar_dir(c):
            print("Using local CIFAR-100 folder:", c); return c
        if os.path.isfile(c) and c.endswith((".tar.gz", ".tgz", ".tar")):
            print("Found local archive:", c)
            d = _extract(c)
            if _is_cifar_dir(d): return d
    print("No local CIFAR-100 found -> downloading (can be slow). Tip: upload cifar-100-python.tar.gz to Drive / a Kaggle dataset once.")
    import torchvision
    torchvision.datasets.CIFAR100(DATA_DIR, train=True, download=True)
    return os.path.join(DATA_DIR, "cifar-100-python")

def load_cifar100():
    d = locate_cifar_dir()
    mean = torch.tensor([0.5071, 0.4865, 0.4409]).view(1, 3, 1, 1); std = torch.tensor([0.2673, 0.2564, 0.2762]).view(1, 3, 1, 1)
    def split(name):
        with open(os.path.join(d, name), "rb") as f:
            e = pickle.load(f, encoding="latin1")
        x = torch.from_numpy(np.asarray(e["data"], dtype=np.uint8).reshape(-1, 3, 32, 32)).float().div(255)
        return ((x - mean) / std).to(dev), torch.tensor(e["fine_labels"]).to(dev)
    return split("train") + split("test")

Xtr, ytr, Xte, yte = load_cifar100()
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))

def augment(x):
    """Random crop (pad 4, reflect) + horizontal flip, per-sample, on GPU."""
    B = x.size(0)
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i = torch.randint(0, 9, (B,), device=x.device); j = torch.randint(0, 9, (B,), device=x.device)
    ar = torch.arange(32, device=x.device)
    rows = (i[:, None] + ar)[:, :, None]; cols = (j[:, None] + ar)[:, None, :]
    x = x[torch.arange(B, device=x.device)[:, None, None], :, rows, cols].permute(0, 3, 1, 2)
    flip = torch.rand(B, device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x).contiguous()

def cutmix(x, y, alpha):
    lam = np.random.beta(alpha, alpha); B, _, H, W = x.shape
    idx = torch.randperm(B, device=x.device)
    rh, rw = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam))
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2, x1, x2 = max(cy - rh // 2, 0), min(cy + rh // 2, H), max(cx - rw // 2, 0), min(cx + rw // 2, W)
    x = x.clone(); x[:, :, y1:y2, x1:x2] = x[idx][:, :, y1:y2, x1:x2]
    return x, y[idx], 1 - (y2 - y1) * (x2 - x1) / (H * W)

In [ ]:
# Evaluation helpers
def resize(x, r):
    if x.shape[-1] == r: return x
    return F.interpolate(x, size=(r, r), mode="bilinear", align_corners=False, antialias=r < x.shape[-1])

@torch.no_grad()
def predict(model, X, res=32, back_to=None, bs=1000):
    model.eval(); outs = []
    for i in range(0, len(X), bs):
        xb = resize(X[i:i + bs], res)
        if back_to: xb = resize(xb, back_to)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            outs.append(model(xb).float())
    return torch.cat(outs)

def metrics(logits, y, nb=15):
    loss = F.cross_entropy(logits, y).item()
    conf, pred = logits.softmax(1).max(1); correct = (pred == y).float()
    top5 = (logits.topk(5, 1).indices == y[:, None]).any(1).float().mean().item() * 100
    edges = torch.linspace(0, 1, nb + 1, device=logits.device); ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): ece += m.float().mean().item() * abs(conf[m].mean().item() - correct[m].mean().item())
    return dict(loss=loss, top1=correct.mean().item() * 100, top5=top5, ece=ece * 100)

@torch.no_grad()
def bench(model, res=32, bs=512, warm=3, iters=10):
    model.eval(); x = torch.randn(bs, 3, res, res, device=dev)
    if dev.type == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
        for _ in range(warm): model(x)
        if dev.type == "cuda": torch.cuda.synchronize()
        t = time.time()
        for _ in range(iters): model(x)
        if dev.type == "cuda": torch.cuda.synchronize()
    dt = time.time() - t
    return bs * iters / dt, (torch.cuda.max_memory_allocated() / 1e9 if dev.type == "cuda" else float("nan"))

def t_sweep(model, X, y, n=TSWEEP_N):
    """Accuracy (native, no adapter) for every (resolution, T). T is overridden in all NCA blocks, inference only."""
    mixers = [b.mix for b in model.blocks]; orig = [m.t_mode for m in mixers]; out = {}
    nominal = {r: model.T_at(r) for r in RES_LIST}                  # read BEFORE overriding t_mode
    Ts = sorted(set(T_LIST) | set(nominal.values()))                # full grid always contains every nominal T
    try:
        for r in RES_LIST:
            out[r] = {}
            for T in Ts:
                for m in mixers: m.t_mode = int(T)
                out[r][T] = metrics(predict(model, X[:n], r, bs=500), y[:n])["top1"]
    finally:
        for m, o in zip(mixers, orig): m.t_mode = o
    return out

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# Training
def run_experiment(spec, seed):
    name, arch, tr = spec["name"], spec["arch"], spec["train"]
    lr0 = LR_PER_MODEL.get(name, CFG["lr"]); train_res = tr.get("train_res")
    h = hashlib.md5(json.dumps([CFG, spec, lr0], sort_keys=True, default=str).encode()).hexdigest()[:6]
    tag = "".join(c if c.isalnum() else "_" for c in name).strip("_") + "_" + h
    path = os.path.join(RUNS, f"{tag}_s{seed}.json")
    if os.path.exists(path):
        print(f"[cached] {name} seed {seed}"); return json.load(open(path))
    set_seed(seed)
    model = build_arch(arch).to(dev)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([dict(params=[p for p in params if p.ndim > 1], weight_decay=CFG["wd"]),
                             dict(params=[p for p in params if p.ndim <= 1], weight_decay=0.0)], lr=lr0)
    try: scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    N, bs, E = len(Xtr), CFG["bs"], CFG["epochs"]; spe = N // bs; total = E * spe; warm = CFG["warmup"] * spe
    lr_at = lambda s: lr0 * (s + 1) / warm if s < warm else 0.5 * lr0 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    T32 = model.T_at(32) if hasattr(model, "T_at") else None
    print(f"\n=== {name} | seed {seed} | params {n_params(model)/1e6:.3f}M | T@32px = {T32} ===\n    {arch_str(arch)}\n    {recipe_str(spec)}")
    if dev.type == "cuda": torch.cuda.reset_peak_memory_stats()
    hist, step, t_train = [], 0, 0.0
    for ep in range(E):
        model.train(); t0 = time.time(); perm = torch.randperm(N, device=dev)
        tl = torch.zeros((), device=dev); tc = torch.zeros((), device=dev)
        for it in range(spe):
            idx = perm[it * bs:(it + 1) * bs]; xb, yb = augment(Xtr[idx]), ytr[idx]
            if train_res: xb = resize(xb, random.choice(train_res))               # per-batch resolution
            lam, yb2 = 1.0, None
            if CFG["cutmix_prob"] > 0 and random.random() < CFG["cutmix_prob"]:
                xb, yb2, lam = cutmix(xb, yb, CFG["cutmix_alpha"])
            for g in opt.param_groups: g["lr"] = lr_at(step)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                out = model(xb)
                loss = F.cross_entropy(out, yb, label_smoothing=CFG["ls"])
                if yb2 is not None:
                    loss = lam * loss + (1 - lam) * F.cross_entropy(out, yb2, label_smoothing=CFG["ls"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(params, CFG["clip"]); scaler.step(opt); scaler.update()
            tl += loss.detach() * len(idx); tc += (out.argmax(1) == yb).sum(); step += 1
        if dev.type == "cuda": torch.cuda.synchronize()
        et = time.time() - t0; t_train += et
        m = metrics(predict(model, Xte), yte)
        hist.append(dict(epoch=ep + 1, train_loss=tl.item() / (spe * bs), train_acc=tc.item() / (spe * bs) * 100,
                         test_loss=m["loss"], test_acc=m["top1"], time=et))
        print(f"ep {ep+1:02d}/{E} | loss {hist[-1]['train_loss']:.3f} | train {hist[-1]['train_acc']:.1f} | test {m['top1']:.2f} | {et:.1f}s"
              + (f" | est. training time {et*E/60:.1f} min" if ep == 0 else ""))
    peak = torch.cuda.max_memory_allocated() / 1e9 if dev.type == "cuda" else float("nan")
    fm = metrics(predict(model, Xte), yte); trm = metrics(predict(model, Xtr[:10000]), ytr[:10000])   # final epoch only
    thr, _ = bench(model, 32)
    res = dict(name=name, role=spec["role"], family=arch["family"], seed=seed, config=arch_str(arch), recipe=recipe_str(spec),
               arch=arch, top1=fm["top1"], top5=fm["top5"], loss=fm["loss"], ece=fm["ece"], train_acc_clean=trm["top1"],
               gap=trm["top1"] - fm["top1"], params_M=n_params(model) / 1e6, gflops=count_flops(model, 32),
               train_min=t_train / 60, sec_per_epoch=t_train / E, throughput=thr, peak_train_mem_gb=peak, T_train=T32,
               history=hist, res_native={}, res_adapt={}, T_by_res={}, tsweep={}, cfg_hash=h)
    for r in RES_LIST:
        res["T_by_res"][r] = model.T_at(r) if hasattr(model, "T_at") else None
    if RUN_RESOLUTION:
        for r in RES_LIST:
            try: res["res_native"][r] = metrics(predict(model, Xte, r), yte)["top1"]
            except Exception: res["res_native"][r] = None          # fixed-N token mixing cannot run at other sizes
            res["res_adapt"][r] = metrics(predict(model, Xte, r, back_to=32), yte)["top1"]
    if RUN_TSWEEP and arch["family"] == "ca" and seed in TSWEEP_SEEDS:
        res["tsweep"] = t_sweep(model, Xte, yte)
    json.dump(res, open(path, "w"))
    if SAVE_CKPT: torch.save(model.state_dict(), os.path.join(CKPT, f"{tag}_s{seed}.pt"))
    print(f"FINAL {name}: top1 {res['top1']:.2f} top5 {res['top5']:.2f} ECE {res['ece']:.2f} | {res['train_min']:.1f} min")
    del model, opt
    if dev.type == "cuda": torch.cuda.empty_cache()
    return json.load(open(path))                                   # round-trip -> fresh and cached runs look identical

all_runs, jobs = [], [(sp, s) for s in SEEDS for sp in ACTIVE]    # seed-major: partial results stay balanced
for k, (sp, s) in enumerate(jobs, 1):
    print(f"\n######## job {k}/{len(jobs)} ########")
    all_runs.append(run_experiment(sp, s))

In [ ]:
# Complexity scaling (random-init models rebuilt per resolution)
cdf = pd.DataFrame()
if RUN_COMPLEXITY:
    rows = []
    for sp in [x for x in ACTIVE if x["group"] == "main"] or ACTIVE:
        for r in COMPLEXITY_RES:
            try:
                m = build_arch(sp["arch"], img=r).to(dev).eval()
                thr, mem = bench(m, r, bs=32, warm=2, iters=5)
                rows.append(dict(model=sp["name"], res=r, tokens=(r // 4) ** 2, params_M=n_params(m) / 1e6, gflops=count_flops(m, r),
                                 img_per_s=thr, infer_mem_gb=mem, T=m.T_at(r) if hasattr(m, "T_at") else None))
                del m
            except Exception as e:
                print("benchmark failed", sp["name"], r, str(e)[:80])
            if dev.type == "cuda": torch.cuda.empty_cache()
    cdf = pd.DataFrame(rows); cdf.to_csv(os.path.join(OUT, "table_complexity.csv"), index=False)
    print(cdf.round(3).to_string(index=False))

In [ ]:
# Aggregate -> tables
runs = [r for r in all_runs if r]
SC = ["seed", "top1", "top5", "loss", "ece", "train_acc_clean", "gap", "params_M", "gflops", "train_min", "sec_per_epoch",
      "throughput", "peak_train_mem_gb", "T_train"]
df = pd.DataFrame([{"name": r["name"], **{k: r.get(k) for k in SC}} for r in runs])
for k in SC: df[k] = pd.to_numeric(df[k], errors="coerce")
order = [sp["name"] for sp in SPECS if sp["name"] in set(df["name"])]
COLS = [("top1", "Top-1 (%)", 2), ("top5", "Top-5 (%)", 2), ("loss", "Test loss", 3), ("ece", "ECE (%)", 2),
        ("gap", "Train-test gap", 2), ("params_M", "Params (M)", 3), ("gflops", "GFLOPs/img", 3), ("sec_per_epoch", "s/epoch", 1),
        ("train_min", "Train (min)", 1), ("throughput", "Img/s", 0), ("peak_train_mem_gb", "Train mem (GB)", 2)]
rows, num = [], []
for nme in order:
    g, sp = df[df.name == nme], SPEC_BY_NAME[nme]
    Tv = g["T_train"].iloc[0]
    row = {"Model": nme, "Role": sp["role"], "Config": arch_str(sp["arch"]), "Recipe": recipe_str(sp), "n_seeds": len(g),
           "T@32px": "-" if pd.isna(Tv) else int(Tv)}
    nr = {"name": nme, "role": sp["role"], "n_seeds": len(g)}
    for k, lab, d in COLS:
        row[lab] = f"{g[k].mean():.{d}f} ± {g[k].std(ddof=1):.{d}f}" if len(g) > 1 else f"{g[k].mean():.{d}f}"
        nr[k + "_mean"] = g[k].mean(); nr[k + "_std"] = g[k].std(ddof=1) if len(g) > 1 else np.nan
    rows.append(row); num.append(nr)
summary, numdf = pd.DataFrame(rows), pd.DataFrame(num)
summary.to_csv(os.path.join(OUT, "summary.csv"), index=False); numdf.to_csv(os.path.join(OUT, "summary_numeric.csv"), index=False)
print(summary.drop(columns=["Config", "Recipe"]).to_string(index=False))
open(os.path.join(OUT, "table_main.tex"), "w").write(to_tex(
    summary[["Model", "Params (M)", "GFLOPs/img", "T@32px", "Top-1 (%)", "Top-5 (%)", "Test loss", "ECE (%)", "Train-test gap"]],
    f"CIFAR-100 results (final epoch, {N_SEEDS} seed(s))."))

# ---- matched comparisons: CA variants vs every baseline ----
CA_NAMES = [n for n in order if SPEC_BY_NAME[n]["arch"]["family"] == "ca" and SPEC_BY_NAME[n]["role"] == "proposed"]
BASELINES = [n for n in order if SPEC_BY_NAME[n]["role"] in ("reference", "param-matched", "FLOP-matched")]
cmp_rows = []
for ca in CA_NAMES:
    a = df[df.name == ca]
    for b_name in BASELINES:
        b = df[df.name == b_name]; p = np.nan
        if len(a) > 1 and len(b) > 1:
            try:
                from scipy.stats import ttest_ind
                p = ttest_ind(a.top1, b.top1, equal_var=False).pvalue
            except Exception: pass
        cmp_rows.append(dict(CA=ca, baseline=b_name, match=SPEC_BY_NAME[b_name]["role"], d_top1=a.top1.mean() - b.top1.mean(),
                             d_top5=a.top5.mean() - b.top5.mean(), d_loss=a.loss.mean() - b.loss.mean(),
                             params_ratio=a.params_M.mean() / b.params_M.mean(), flops_ratio=a.gflops.mean() / b.gflops.mean(),
                             throughput_ratio=a.throughput.mean() / b.throughput.mean(), train_time_ratio=a.train_min.mean() / b.train_min.mean(),
                             p_top1=p, n_seeds=f"{len(a)} vs {len(b)}"))
cmpdf = pd.DataFrame(cmp_rows); cmpdf.to_csv(os.path.join(OUT, "table_matched.csv"), index=False)
print("\nCA-Mixer minus baseline (positive d_top1 = CA-Mixer better):")
print(cmpdf.round(3).to_string(index=False) if len(cmpdf) else "(need at least one CA-Mixer and one baseline)")
print(f"\nPILOT READ-OUT (n_seeds={N_SEEDS})")
for r in cmpdf[cmpdf.CA.str.contains("spec")].itertuples() if len(cmpdf) else []:
    print(f"  spec CA-Mixer vs {r.baseline}: d_top1 {r.d_top1:+.2f} | FLOPs x{r.flops_ratio:.2f} | params x{r.params_ratio:.2f} | speed x{r.throughput_ratio:.2f}")
if N_SEEDS == 1:
    print("  Single seed: the earlier 3-seed runs had std ~0.2-0.6 pts, so treat |d_top1| < ~1.5 pts as inconclusive.\n"
          "  If CA-Mixer still leads the matched baselines by more than that, set N_SEEDS = 5 and re-run (finished runs are cached).")

In [ ]:
# Resolution + T-sweep tables
rr = []
for r in runs:
    for k, v in r["res_native"].items():
        rr.append(dict(name=r["name"], seed=r["seed"], res=int(k), mode="native", top1=v, T=(r.get("T_by_res") or {}).get(k)))
    for k, v in r["res_adapt"].items():
        rr.append(dict(name=r["name"], seed=r["seed"], res=int(k), mode="resize->32", top1=v, T=r.get("T_train")))
rdf = pd.DataFrame(rr)
if len(rdf):
    for c in ("top1", "T"): rdf[c] = pd.to_numeric(rdf[c], errors="coerce")
    rt = rdf.groupby(["name", "mode", "res"]).agg(top1_mean=("top1", "mean"), top1_std=("top1", "std"), T=("T", "first")).reset_index()
    rt.to_csv(os.path.join(OUT, "table_resolution.csv"), index=False)
    print(rt[rt["mode"] == "native"].round(2).to_string(index=False))

ts = []
for r in runs:
    for k, d in (r.get("tsweep") or {}).items():
        for T, v in d.items():
            ts.append(dict(name=r["name"], seed=r["seed"], res=int(k), T=int(T), top1=v, T_nominal=(r.get("T_by_res") or {}).get(k)))
tsdf = pd.DataFrame(ts)
if len(tsdf):
    tsm = tsdf.groupby(["name", "res", "T"]).agg(top1=("top1", "mean"), T_nominal=("T_nominal", "first")).reset_index()
    tsm.to_csv(os.path.join(OUT, "table_tsweep.csv"), index=False)
    best = []
    for (nme, rs), g in tsm.groupby(["name", "res"]):
        Tn = g["T_nominal"].iloc[0]; gn = g[g["T"] == Tn]; b = g.loc[g["top1"].idxmax()]
        best.append(dict(name=nme, res=rs, T_nominal=Tn, acc_at_nominal=gn["top1"].iloc[0] if len(gn) else np.nan,
                         best_T=b["T"], acc_at_best=b["top1"], gain=b["top1"] - (gn["top1"].iloc[0] if len(gn) else np.nan)))
    bdf = pd.DataFrame(best); bdf.to_csv(os.path.join(OUT, "table_best_T.csv"), index=False)
    print("\nBest T per resolution (T-sweep, first %d test images):" % TSWEEP_N); print(bdf.round(2).to_string(index=False))

In [ ]:
# Figures
FAM_MARK = {"mixer": "s", "resmlp": "^", "ca": "o", "dwconv": "D"}
main_names = [n for n in order if SPEC_BY_NAME[n]["group"] in ("main", "ca_extra")] or order
cmap = plt.get_cmap("tab10")

# 1) training curves
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for i, n in enumerate(main_names):
    for a_, key, lab in ((ax[0], "train_loss", "Train loss"), (ax[1], "test_acc", "Test top-1 (%)")):
        H = np.array([[h[key] for h in r["history"]] for r in runs if r["name"] == n]); ep = np.arange(1, H.shape[1] + 1)
        a_.plot(ep, H.mean(0), color=cmap(i % 10), ls="-" if SPEC_BY_NAME[n]["arch"]["family"] == "ca" else "--", label=n)
        if len(H) > 1: a_.fill_between(ep, H.mean(0) - H.std(0), H.mean(0) + H.std(0), color=cmap(i % 10), alpha=.15)
        a_.set_ylabel(lab); a_.set_xlabel("Epoch"); a_.grid(alpha=.3)
ax[1].legend(fontsize=7); plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_training_curves.png"), dpi=200); plt.show()

# 2) accuracy vs params / GFLOPs
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for _, r in numdf.iterrows():
    fam = SPEC_BY_NAME[r["name"]]["arch"]["family"]
    for a_, xk in ((ax[0], "params_M_mean"), (ax[1], "gflops_mean")):
        a_.scatter(r[xk], r["top1_mean"], marker=FAM_MARK[fam], s=60, color=cmap(list(numdf["name"]).index(r["name"]) % 10))
        a_.annotate(r["name"].replace("CA-Mixer", "CA").replace("(", "").replace(")", ""), (r[xk], r["top1_mean"]), fontsize=6,
                    xytext=(3, 3), textcoords="offset points")
ax[0].set_xlabel("Parameters (M)"); ax[1].set_xlabel("GFLOPs / image"); ax[1].set_xscale("log")
for a_ in ax: a_.set_ylabel("Top-1 (%)"); a_.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_pareto.png"), dpi=200); plt.show()

# 3) resolution robustness
if len(rdf):
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
    for i, n in enumerate(order):
        for a_, mode in ((ax[0], "native"), (ax[1], "resize->32")):
            g = rdf[(rdf.name == n) & (rdf["mode"] == mode)].dropna(subset=["top1"])
            if len(g):
                s = g.groupby("res").top1.agg(["mean", "std"]).fillna(0)
                a_.errorbar(s.index, s["mean"], yerr=s["std"], marker="o", capsize=2, color=cmap(i % 10), label=n, lw=1.2)
    ax[0].set_title("Native (model runs at the test resolution; fixed-N models cannot)"); ax[1].set_title("Resize-to-32 adapter (all models)")
    for a_ in ax: a_.set_xlabel("Test resolution (trained at 32x32)"); a_.set_ylabel("Top-1 (%)"); a_.grid(alpha=.3)
    ax[1].legend(fontsize=6); plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_resolution.png"), dpi=200); plt.show()

# 4) T-sweep heatmaps (accuracy for every resolution x T; dot = nominal T of the T function)
if len(tsdf):
    names = [n for n in main_names if n in set(tsm["name"])][:4]
    Tall = sorted(tsm["T"].unique()); Rall = sorted(tsm["res"].unique())
    fig, ax = plt.subplots(1, len(names), figsize=(4.6 * len(names), 4), squeeze=False)
    for a_, n in zip(ax[0], names):
        g = tsm[tsm.name == n]; M = np.full((len(Rall), len(Tall)), np.nan)
        for _, r in g.iterrows(): M[Rall.index(r["res"]), Tall.index(r["T"])] = r["top1"]
        a_.imshow(M, cmap="viridis", aspect="auto", vmin=0, vmax=max(1.0, np.nanmax(M)))
        for i in range(len(Rall)):
            for j in range(len(Tall)):
                if not np.isnan(M[i, j]): a_.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center", color="w", fontsize=7)
        for i, rs in enumerate(Rall):
            Tn = g[g.res == rs]["T_nominal"].iloc[0]
            if Tn in Tall: a_.plot(Tall.index(Tn), i, "o", mfc="none", mec="red", ms=14)
        a_.set_xticks(range(len(Tall))); a_.set_xticklabels(Tall); a_.set_yticks(range(len(Rall))); a_.set_yticklabels(Rall)
        a_.set_xlabel("T (steps per block)"); a_.set_ylabel("Test resolution"); a_.set_title(n.replace("CA-Mixer ", "CA "), fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_tsweep.png"), dpi=200); plt.show()

# 5) complexity scaling
if len(cdf):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for i, (n, g) in enumerate(cdf.dropna(subset=["gflops"]).groupby("model", sort=False)):
        fam = SPEC_BY_NAME[n]["arch"]["family"]
        for a_, k in zip(ax, ("gflops", "img_per_s", "params_M")):
            a_.plot(g.tokens, g[k], marker=FAM_MARK[fam], color=cmap(i % 10), label=n)
    for a_, t in zip(ax, ["GFLOPs / image", "Inference img/s (bs=32)", "Parameters (M)"]):
        a_.set_xscale("log"); a_.set_yscale("log"); a_.set_xlabel("Number of tokens N"); a_.set_title(t); a_.grid(alpha=.3)
    ax[0].legend(fontsize=6); plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_complexity.png"), dpi=200); plt.show()

In [ ]:
# Zip results for download
zp = shutil.make_archive(os.path.join(os.path.dirname(OUT.rstrip("/")) or ".", RESULTS_FOLDER + "_zip"), "zip", OUT)
print("Saved:", zp)
# Colab: from google.colab import files; files.download(zp)    (or just copy from Drive)
# Kaggle: the folder and zip in /kaggle/working appear in the notebook's Output tab after "Save Version".